# 05 · Chapter 3: The whole support office

Every message goes to the **cheapest helper that can answer it**:

```
message → Receptionist (ML model) ──at least 80% sure + simple──→ ready-made answer      ($0, ~0.05 ms)
              │
              └─40-80% sure, policy question──→ Junior assistant (local LLM)          ($0)
              │
              └─action / complaint / unsure──→ Senior expert (cloud LLM + tools)       (paid)
                                                      │
                                                      └─big refund / very upset──→ Manager (a human)
```

**My plan:**
1. See who would answer each message (free, no LLM)
2. Try one example per helper
3. Compare the expert alone (chapter 1) with the whole office on a mixed inbox

In [1]:
import sys
sys.path.append("..")   # so Python can find the budget_agent folder

import pandas as pd
pd.set_option("display.max_colwidth", 80)
from dotenv import load_dotenv
load_dotenv("../.env")

from budget_agent import config, data
from budget_agent.router import Router

office = Router()
print("Junior assistant (Ollama) ready:", office.local_ready())

Junior assistant (Ollama) ready: True


## Step 1: Who would answer? (no LLM is called here)

In [2]:
for helper, message in config.DEMO_MESSAGES.items():
    layer, intent, confidence = office.choose_layer(message)
    print(f"{config.HELPERS[layer]:32} {intent:24} {confidence:4.0%}   {message}")

Receptionist (ML model)          track_order               93%   track my order NM10009
Junior assistant (local LLM)     delivery_options          57%   kitne din me delivery hoti hai
Senior expert (cloud LLM agent)  track_order               52%   my order NM10002 arrived broken, I want a refund
Senior expert (cloud LLM agent)  get_refund                27%   refund NM10018, I don't like the chair


The manager's message also goes to the senior expert first. The expert then hands it to the manager, because the guardrail in `tools.py` says refunds above ₹10,000 need a human.

If Ollama isn't running, the junior's message goes to the senior expert instead.

### The routing rule, written by hand

The real rule is `Router.choose_layer()` in `budget_agent/router.py`. Here it is in a few lines, so I can see every decision:

In [3]:
from budget_agent import classifier
from budget_agent.router import ORDER_INTENTS   # track_order, track_refund: they need an order lookup

def who_answers(text, junior_ready):
    intent, sure = classifier.predict(office.model, text)
    simple = intent not in config.ACTION_INTENTS                 # refunds, cancels... are never simple
    if simple and sure >= config.ML_CONFIDENCE:                  # 0.80
        return "ml"                                              # receptionist
    if simple and intent not in ORDER_INTENTS and sure >= config.LOCAL_LLM_CONFIDENCE and junior_ready:  # 0.40
        return "local_llm"                                       # junior: policy questions only
    return "cloud_llm"                                           # senior expert

test = data.load_messages().query("split == 'test'")
agrees = all(who_answers(t, office.local_ready()) == office.choose_layer(t)[0] for t in test.text)
print("My rule gives the same answer as router.py on all", len(test), "test messages:", agrees)

My rule gives the same answer as router.py on all 540 test messages: True


In [4]:
helpers = [config.HELPERS[office.choose_layer(text)[0]] for text in test.text]
pd.Series(helpers).value_counts(normalize=True)

Receptionist (ML model)            0.520370
Senior expert (cloud LLM agent)    0.348148
Junior assistant (local LLM)       0.131481
Name: proportion, dtype: float64

On normal messages, a big share never needs the expert at all.
(Refunds, cancels, complaints etc. always go to the expert, on purpose, because they change real things.)

## Step 2: One example per helper

In [5]:
def show(r):
    who = config.HELPERS[r.layer]
    if r.escalated:                                   # the expert handed it to a human
        who += " → " + config.HELPERS["human"]
    print(f"[{who}] intent={r.intent} ({r.confidence:.0%})  {r.seconds:.3f}s  ${r.cost_usd:.4f}")
    print("  ", r.reply, "\n")

show(office.handle(config.DEMO_MESSAGES["ml"]))          # receptionist

[Receptionist (ML model)] intent=track_order (93%)  0.001s  $0.0000
   Your order NM10009 (Running Shoes) is Shipped. Delivery date: 2026-10-08. 



In [6]:
show(office.handle(config.DEMO_MESSAGES["local_llm"]))   # junior assistant (if Ollama is running)

[Junior assistant (local LLM)] intent=delivery_options (57%)  1.163s  $0.0000
   Standard delivery mein 3-5 kaamre din, Express delivery mein 1-2 kaamre din hoti hai. 



In [7]:
show(office.handle(config.DEMO_MESSAGES["cloud_llm"]))   # senior expert

[Senior expert (cloud LLM agent)] intent=track_order (52%)  4.127s  $0.0029
   Sorry about that — I’ve started the refund for order NM10002. The ₹5,999 should go back to your original payment method in 5–7 working days. 



In [8]:
show(office.handle(config.DEMO_MESSAGES["human"]))       # senior expert → manager

[Senior expert (cloud LLM agent) → Manager (human)] intent=get_refund (27%)  4.089s  $0.0030
   I’ve checked your order, and it’s eligible for a refund since it was delivered on 02 Oct 2026. Because the amount is ₹12,999, a human agent needs to approve it, and I’ve escalated this for you. 



## Step 3: The expert alone vs the whole office, on a mixed inbox

This calls the cloud LLM many times, so I start small (`--n 30` costs well under $1).
I run it from the terminal so it saves a CSV the app can show:

```
uv run python run.py benchmark --n 30
```

In [9]:
results = pd.read_csv(config.DATA_DIR / "benchmark_results.csv")
summary = pd.DataFrame({
    "total cost ($)": [results.v2_cost.sum(), results.v4_cost.sum()],
    "average seconds": [results.v2_seconds.mean(), results.v4_seconds.mean()],
}, index=["Expert only (chapter 1)", "Whole office (chapter 3)"])
summary.round(3)

,total cost ($),average seconds
Expert only (chapter 1),0.059,1.701
Whole office (chapter 3),0.031,1.040


In [10]:
print(f"The whole office is {results.v2_cost.sum() / results.v4_cost.sum():.1f}x cheaper")
results.v4_layer.map(config.HELPERS).value_counts()

The whole office is 1.9x cheaper


v4_layer
Senior expert (cloud LLM agent)    27
Receptionist (ML model)            26
Junior assistant (local LLM)        7
Name: count, dtype: int64

In [11]:
# Read some replies side by side, to check quality didn't drop
results[["message", "v4_layer", "v2_reply", "v4_reply"]].sample(5, random_state=1)

,message,v4_layer,v2_reply,v4_reply
39,help to cancel purchase NM10108,cloud_llm,"I checked NM10108, and it’s already **Shipped**, so I can’t cancel it now. I...","I checked NM10108, and it’s already **Shipped**, so I can’t cancel it now. I..."
41,cqn i swap several items of purchase NM10009,cloud_llm,"I checked order NM10009, and it’s already shipped, so we can’t swap or chang...","I checked order NM10009, and it’s already shipped, so we can’t swap or chang..."
2,"wheres my stuf, order no NM10101",cloud_llm,"Your order NM10101 is marked as cancelled, so it won’t be delivered. If you ...","Your order NM10101 is marked as cancelled, so it won’t be delivered. If you ..."
48,download invoices from Rahul,ml,"Please share the order number for Rahul’s order, and I can help check the in...",You can download your invoice from My Orders on the website. It is also emai...
50,I want assistance to set the goddamn delivery address up,ml,"Sure — I can help update it, but I need your order number first. Please send...","You can add a shipping address under Account → Addresses, or at checkout."


## What I learned
- Most of a support inbox is simple. The receptionist (a tiny ML model) handles it in milliseconds, for free.
- The office only pays for the senior expert when it's really needed: actions, complaints, unclear messages.
- The 80% rule is a **dial**: higher = safer but more expensive, lower = cheaper but riskier.
- This is **inference engineering**: the right model for each request, not one giant model for everything.